Loading the dataset

In [2]:
import pandas as pd
import numpy as np

file_path = "data/FreshChoice_Analytics_Project_Dataset.xlsx"

sales = pd.read_excel(file_path, sheet_name="Sales")

products = pd.read_excel(file_path, sheet_name="Products")

stores = pd.read_excel(file_path, sheet_name="Stores")

customers = pd.read_excel(file_path, sheet_name="Customers")

calendar = pd.read_excel(file_path, sheet_name="Calendar")

targets = pd.read_excel(file_path, sheet_name="Targets")

customer_features = pd.read_excel(
    file_path,
    sheet_name="Customer_Features"
)

print(sales.head())

    Sales_ID  Invoice_No Invoice_Date Store_ID Product_ID Customer_ID  \
0  SL0000001  INV0000001   2025-04-22     S007      P0159      C00427   
1  SL0000002  INV0000001   2025-04-22     S007      P0129      C00427   
2  SL0000003  INV0000001   2025-04-22     S007      P0057      C00427   
3  SL0000004  INV0000002   2025-04-09     S008      P0122      C01512   
4  SL0000005  INV0000003   2025-11-12     S009      P0127      C00438   

   Quantity  Unit_Price  Discount_Amount  Total_Sales  Cost_Per_Unit  \
0         3      655.62             0.00      1966.87         551.70   
1         1      263.15             0.00       263.15         176.66   
2         3      551.32           165.40      1488.55         353.15   
3         3     1152.85             0.00      3458.54         851.21   
4         3      806.15            72.55      2345.90         484.67   

  Payment_Method  Line_Cost  Gross_Sales  Profit  
0           Cash    1655.10      1966.86  311.77  
1           Cash     176.6

Inspect the data

In [3]:
print("Dataset shape:", sales.shape)

print("\nColumn names:")
print(sales.columns.tolist())

print("\nData types:")
print(sales.dtypes)

print("\nSummary statistics:")
print(sales.describe())

print("\nMissing values:")
print(sales.isnull().sum())

Dataset shape: (45324, 15)

Column names:
['Sales_ID', 'Invoice_No', 'Invoice_Date', 'Store_ID', 'Product_ID', 'Customer_ID', 'Quantity', 'Unit_Price', 'Discount_Amount', 'Total_Sales', 'Cost_Per_Unit', 'Payment_Method', 'Line_Cost', 'Gross_Sales', 'Profit']

Data types:
Sales_ID                      str
Invoice_No                    str
Invoice_Date       datetime64[us]
Store_ID                      str
Product_ID                    str
Customer_ID                   str
Quantity                    int64
Unit_Price                float64
Discount_Amount           float64
Total_Sales               float64
Cost_Per_Unit             float64
Payment_Method                str
Line_Cost                 float64
Gross_Sales               float64
Profit                    float64
dtype: object

Summary statistics:
                     Invoice_Date      Quantity    Unit_Price  \
count                       45324  45324.000000  45324.000000   
mean   2025-07-06 15:30:46.227164      1.978532    71

Check for duplicate records

In [4]:
print(
    "Duplicate Sales_ID:",
    sales["Sales_ID"].duplicated().sum()
)

print(
    "Duplicate full rows:",
    sales.duplicated().sum()
)

Duplicate Sales_ID: 0
Duplicate full rows: 0


Check and correct the data types

In [5]:
#convert the transaction date to a proper datetime field
sales["Invoice_Date"] = pd.to_datetime(
    sales["Invoice_Date"],
    errors="coerce"
)

In [6]:
#convert nuumerical fields
numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Discount_Amount",
    "Total_Sales",
    "Cost_Per_Unit",
    "Line_Cost",
    "Gross_Sales",
    "Profit"
]

for column in numeric_columns:
    sales[column] = pd.to_numeric(
        sales[column],
        errors="coerce"
    )

In [7]:
#check the results
print(sales.dtypes)

Sales_ID                      str
Invoice_No                    str
Invoice_Date       datetime64[us]
Store_ID                      str
Product_ID                    str
Customer_ID                   str
Quantity                    int64
Unit_Price                float64
Discount_Amount           float64
Total_Sales               float64
Cost_Per_Unit             float64
Payment_Method                str
Line_Cost                 float64
Gross_Sales               float64
Profit                    float64
dtype: object


Investigate missing values

In [8]:
missing = sales.isnull().sum()

missing = missing[missing > 0].sort_values(
    ascending=False
)

print(missing)

Series([], dtype: int64)


Check for invalid values

In [9]:
invalid_sales = sales[
    (sales["Quantity"] <= 0)
    | (sales["Unit_Price"] < 0)
    | (sales["Total_Sales"] < 0)
]

print("Potentially invalid records:", len(invalid_sales))

print(invalid_sales.head())

Potentially invalid records: 0
Empty DataFrame
Columns: [Sales_ID, Invoice_No, Invoice_Date, Store_ID, Product_ID, Customer_ID, Quantity, Unit_Price, Discount_Amount, Total_Sales, Cost_Per_Unit, Payment_Method, Line_Cost, Gross_Sales, Profit]
Index: []


Create additional features

In [10]:
sales["Gross_Sales_Calculated"] = (
    sales["Quantity"] * sales["Unit_Price"]
)

sales["Net_Sales_Calculated"] = (
    sales["Gross_Sales_Calculated"]
    - sales["Discount_Amount"]
)

sales["Line_Cost_Calculated"] = (
    sales["Quantity"] * sales["Cost_Per_Unit"]
)

sales["Profit_Calculated"] = (
    sales["Net_Sales_Calculated"]
    - sales["Line_Cost_Calculated"]
)

In [11]:
#create calendar features
sales["Year"] = sales["Invoice_Date"].dt.year

sales["Month_Number"] = sales["Invoice_Date"].dt.month

sales["Month_Name"] = sales["Invoice_Date"].dt.month_name()

sales["Day_Name"] = sales["Invoice_Date"].dt.day_name()

sales["Is_Weekend"] = (
    sales["Invoice_Date"].dt.dayofweek >= 5
)

Investigate outliers

In [12]:
#use the interquartile range to flag unusually high sales values
q1 = sales["Total_Sales"].quantile(0.25)

q3 = sales["Total_Sales"].quantile(0.75)

iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr

outliers = sales[
    sales["Total_Sales"] > upper_limit
]

print("Potential high-value sales lines:", len(outliers))

Potential high-value sales lines: 2071


Save the cleaned dataset

In [13]:
import os

os.makedirs("data/processed", exist_ok=True)

sales.to_csv(
    "data/processed/Sales_Clean.csv",
    index=False
)

print("Cleaned data saved successfully.")

Cleaned data saved successfully.
